# Strategy Backtest v4 — trade-level diagnosis, sector cap and exit rules (walk-forward)

Question from the user: sells often happen below the buy price — is the weekly rank approach (and the 4-per-sector cap) right?

Same engine and honesty rules as v3: point-in-time features, decision at the close, fill at the next open, 0.1% cost per side,
rolling walk-forward from **2022-04-01** (parameters, where a candidate has one, are re-picked every quarter on the trailing
12 months by Sharpe), a **never-seen** segment 2022-04-01 → 2024-09-16, and C6 (the live rule) as the baseline. Bars come from
the local cache (no API calls).

**Pre-declared candidates** (everything else = C6: score 0.5·Technical + 0.5·RS, weekly decision, top 10, score > 0,
inverse-vol weights, soft QQQ regime ×0.5):

| group | id | rule |
|---|---|---|
| sector cap | CAP2 / CAP3 / C6 (4) / CAP5 / CAP-none | max names per sector 2 / 3 / 4 / 5 / 10 (no cap) |
| exits | E1-E3 | hysteresis: enter in top 10 (after caps), hold until rank > 15 / 20 / 25 or score ≤ 0 |
| | E-BUF-WF | buffer ∈ {15, 20, 25} picked walk-forward |
| | E4 / E5 | score exit only: hold until score ≤ 0 / ≤ 10 (entry also needs score > that), refill empty slots by rank |
| | E-SC-WF | score threshold ∈ {0, 10} picked walk-forward |
| | E6 | minimum holding 4 weekly decisions (unless score ≤ 0), otherwise C6 |
| | E7 / E8 | rebalance every 2 weeks / monthly, buffer 20 |
| | E9 / E10 | entry by rank, exit only by a stop checked daily: close < 50-day MA / close < peak since entry − 3×ATR (or score ≤ 0 at a decision); freed slots refilled at the next weekly decision |
| | E-STOP-WF | E9 vs E10 picked walk-forward |
| | E11 | absolute threshold (old style, checked daily): hold while score > X, max 10 by score, X ∈ {10, 20, 30, 40} picked walk-forward |

**Decision rule (pre-declared, unchanged):** switch only if a candidate beats C6 on stitched Sharpe AND its max DD is better or
within 2 points AND its never-seen Sharpe ≥ C6's. A candidate that materially cuts turnover (≥ 25% lower) with Sharpe within
0.03 of C6 and a better max DD is reported as a close call — no switch.

**Bias warning:** the universe was hand-picked in 2026; absolute returns are inflated. Compare candidates with C6 (same bias).

### Setup
Load cached bars, scores and sectors for the C6 diagnosis.

In [1]:
import math
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import backtest_engine as be
from sector_mapping import symbol_sector

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_rows", 300, "display.max_columns", 60, "display.width", 250)
bars, dropped_partial = be.load_bars(refresh=False, cache_name="bars_daily_long.pkl", start="2020-06-01")
tech = be.build_technical(bars)
close_all, open_all = be.wide(bars, "Close"), be.wide(bars, "Open")
syms = [s for s in be.TRADABLE if s in close_all.columns]
idx = close_all.index
score_tech = be.wide(tech, "Technical_Score").reindex(index=idx, columns=syms)
elig = be.bool_wide(tech, "eligible", idx, syms)
atr_w = be.wide(tech, "atr").reindex(index=idx, columns=syms)
ma50_w = be.wide(tech, "ma_50").reindex(index=idx, columns=syms)
rs, _ = be.relative_strength(close_all, syms)
score = 0.5 * score_tech + 0.5 * rs
vol63 = close_all[syms].pct_change(fill_method=None).rolling(63).std()
regime = be.regime_series(close_all, "QQQ")
weekly = be.weekly_rebalance_days(idx, live=True)
biweekly = weekly & (weekly.cumsum() % 2 == 0)
monthly = be.monthly_rebalance_days(idx)
if be.next_sessions(idx[-1], 1)[0].month == idx[-1].month:   # the current month is not finished
    monthly.iloc[-1] = False
daily = pd.Series(True, index=idx)
WF_START, SEEN_START, NEVER_END = "2022-04-01", "2024-09-17", "2024-09-16"
SEGMENTS = {"WF stitched": (WF_START, None), "Never-seen 2022-04→2024-09": (WF_START, NEVER_END),
            "Seen 2024-09→now": (SEEN_START, None), "Last 12m": ((idx[-1] - pd.DateOffset(years=1)).strftime("%Y-%m-%d"), None)}
print(f"Bars {idx.min():%Y-%m-%d} → {idx.max():%Y-%m-%d} | partial bar dropped: {dropped_partial} | {len(syms)} tradable")

Bars 2020-06-01 → 2026-09-24 | partial bar dropped: False | 78 tradable


### Selector (generalises the engine's rank_targets: buffers, score exits, minimum hold, stops) and a trade ledger

In [2]:
COLS = syms
S = score.to_numpy(float)
TB = rs.reindex(index=idx, columns=COLS).to_numpy(float)
E = elig.reindex(index=idx, columns=COLS).astype("boolean").fillna(False).to_numpy(bool)
V = vol63.reindex(index=idx, columns=COLS).to_numpy(float)
CL = close_all[COLS].to_numpy(float)
ATR = atr_w.to_numpy(float)
MA50 = ma50_w.to_numpy(float)
RG = regime.reindex(idx).astype("boolean").fillna(False).to_numpy(bool)
SECT = np.array([symbol_sector.get(c, "Other") for c in COLS])
NAME_POS = be._name_positions(COLS)


def select(decide=weekly, n=10, max_sector=4, entry_min=0.0, keep_rank=None, min_hold=0, stop=None, atr_k=3.0,
           regime_scale=0.5, log=None):
    """Target weights decided at each close.
    decide: decision days. keep_rank: a holding is kept while its rank among qualifying names <= keep_rank (np.inf = any rank)
    and its score > entry_min. min_hold: a holding younger than this many decisions is kept regardless of rank (score > 0).
    stop: None | 'ma50' | 'atr' - checked every close while held; exit next open, slot refilled at the next decision.
    log: list -> one dict per exit decision (Date = decision close, Reason)."""
    D = decide.reindex(idx).astype("boolean").fillna(False).to_numpy(bool)
    out = np.zeros((len(idx), len(COLS)))
    cur, age, peak = np.zeros(len(COLS)), np.zeros(len(COLS), int), np.full(len(COLS), np.nan)
    for t in range(len(idx)):
        if D[t]:
            base_ok = E[t] & ~np.isnan(S[t]) & ~np.isnan(V[t]) & (V[t] > 0)
            ok = base_ok & (S[t] > entry_min)
            order = be.ranking_order(np.flatnonzero(ok), S[t], TB[t], NAME_POS)
            rank_of = {j: r + 1 for r, j in enumerate(order)}
            picked, per, why = [], {}, {}
            held = be.ranking_order(np.flatnonzero(cur > 0), S[t], TB[t], NAME_POS)
            for j in held:
                if not base_ok[j]:
                    continue
                young = min_hold and age[j] < min_hold and S[t, j] > 0
                by_rank = keep_rank is not None and S[t, j] > entry_min and rank_of.get(j, np.inf) <= keep_rank
                if (young or by_rank) and len(picked) < n and per.get(SECT[j], 0) < max_sector:
                    picked.append(j); per[SECT[j]] = per.get(SECT[j], 0) + 1; why[j] = "kept"
            for j in order:
                if len(picked) >= n:
                    break
                if j in why:
                    continue
                if per.get(SECT[j], 0) >= max_sector:
                    why[j] = "cap"; continue
                picked.append(j); per[SECT[j]] = per.get(SECT[j], 0) + 1; why[j] = "selected"
            new = np.zeros(len(COLS))
            if picked:
                inv = 1 / V[t, picked]
                new[picked] = inv / inv.sum() * (len(picked) / n)
                if not RG[t]:
                    new *= regime_scale
            if log is not None:
                for j in np.flatnonzero((cur > 0) & (new == 0)):
                    if not base_ok[j]:
                        r = "not eligible / no data"
                    elif S[t, j] <= max(entry_min, 0.0):
                        r = f"score <= {max(entry_min, 0):g}"
                    elif why.get(j) == "cap":
                        r = "sector cap displacement"
                    else:
                        r = "rank drift (score > 0)"
                    log.append({"Date": idx[t], "Symbol": COLS[j], "Reason": r, "Rank": rank_of.get(j, np.nan), "Score": S[t, j]})
            age = np.where(new > 0, np.where(cur > 0, age + 1, 1), 0)
            peak = np.where(new > 0, np.where(cur > 0, peak, CL[t]), np.nan)
            cur = new
        if stop is not None and (cur > 0).any():
            peak = np.where(cur > 0, np.fmax(peak, CL[t]), np.nan)
            hit = (cur > 0) & ((CL[t] < MA50[t]) if stop == "ma50" else (CL[t] < peak - atr_k * ATR[t]))
            hit &= ~np.isnan(CL[t])
            for j in np.flatnonzero(hit):
                if log is not None:
                    log.append({"Date": idx[t], "Symbol": COLS[j], "Reason": f"stop ({stop})", "Rank": np.nan, "Score": S[t, j]})
            cur = np.where(hit, 0.0, cur); age[hit] = 0; peak[hit] = np.nan
        out[t] = cur
    return pd.DataFrame(out, index=idx, columns=COLS)


def full(t):
    return t.reindex(index=idx, columns=close_all.columns).fillna(0.0)


def sim(target, reb, s, e=None):
    return be.simulate(open_all, close_all, target, s, e, rebalance=reb)


def ledger(target, reb, s, e=None, cost=be.COST):
    """be.simulate plus per-position cash flows -> round trips with P&L in portfolio terms."""
    dates = open_all.index
    s0 = dates.searchsorted(pd.Timestamp(s))
    s1 = len(dates) if e is None else dates.searchsorted(pd.Timestamp(e), side="right")
    cols = list(open_all.columns)
    O = open_all.to_numpy(float)
    Cc = close_all.reindex(columns=cols).ffill().to_numpy(float)
    W = target.reindex(index=dates, columns=cols).fillna(0.0).to_numpy(float)
    R = reb.reindex(dates).astype("boolean").fillna(False).to_numpy(bool) if reb is not None else np.zeros(len(dates), bool)
    N = len(cols)
    sh, basis, flow = np.zeros(N), np.zeros(N), np.zeros(N)
    ent, ent_px, ent_eq = np.full(N, -1), np.zeros(N), np.zeros(N)
    cash, eq, rows = 1.0, [], []
    for t in range(s0, s1):
        o = O[t]
        px = np.where(np.isnan(o), Cc[t - 1], o)
        Vt = cash + np.nansum(sh * np.nan_to_num(px))
        w = W[t - 1]
        trad = ~np.isnan(o)
        if R[t - 1]:
            want = np.where(trad, w * Vt / np.where(trad, o, 1.0), sh)
        else:
            want = sh.copy()
            op, clz = trad & (sh == 0) & (w > 0), trad & (sh > 0) & (w <= 0)
            want[op] = w[op] * Vt / o[op]; want[clz] = 0.0
        d = want - sh
        d[np.abs(d * np.nan_to_num(px)) < 1e-10] = 0.0
        for j in np.where(d < 0)[0]:
            q = -d[j]; cash += q * o[j] * (1 - cost); flow[j] += q * o[j] * (1 - cost)
            if want[j] == 0:
                rows.append({"Symbol": cols[j], "Entry": dates[ent[j]], "Exit": dates[t], "Entry_Open": ent_px[j], "Exit_Open": o[j],
                             "Return_net_%": (o[j] * (1 - cost) / basis[j] - 1) * 100, "PnL_%_of_equity_at_entry": flow[j] / ent_eq[j] * 100,
                             "Exit_Value_%_equity": q * o[j] / Vt * 100, "Sessions": t - ent[j]})
                basis[j] = 0.0; ent[j] = -1; flow[j] = 0.0
            sh[j] = want[j]
        buys = np.where(d > 0)[0]
        need = float(np.sum(d[buys] * o[buys] * (1 + cost))) if len(buys) else 0.0
        scale = min(1.0, max(cash, 0.0) / need) if need > 0 else 1.0
        for j in buys:
            q = d[j] * scale
            if q <= 0:
                continue
            cash -= q * o[j] * (1 + cost); flow[j] -= q * o[j] * (1 + cost)
            new = sh[j] + q
            basis[j] = (basis[j] * sh[j] + q * o[j] * (1 + cost)) / new
            if sh[j] == 0:
                ent[j], ent_px[j], ent_eq[j] = t, o[j], Vt
            sh[j] = new
        eq.append(cash + np.nansum(sh * Cc[t]))
    tr = pd.DataFrame(rows)
    n_open = int((sh > 0).sum())
    return tr, pd.Series(eq, index=dates[s0:s1]), n_open


# sanity: the selector reproduces the engine for C6 and for a rank buffer
c6_log = []
T_C6 = select(log=c6_log)
eng = be.rank_targets(score, elig, vol63, n=10, regime=regime, rebalance_days=weekly, sector_cap=0.4, regime_scale=0.5, tiebreak_w=rs)
eng15 = be.rank_targets(score, elig, vol63, n=10, regime=regime, rebalance_days=weekly, sector_cap=0.4, regime_scale=0.5,
                        buffer_rank=15, tiebreak_w=rs)
print("selector == engine (C6):", np.allclose(T_C6.to_numpy(), eng.reindex(columns=COLS).to_numpy()),
      "| (buffer 15):", np.allclose(select(keep_rank=15).to_numpy(), eng15.reindex(columns=COLS).to_numpy()))
tr_chk, eq_chk, _ = ledger(full(T_C6), weekly, WF_START)
print("ledger equity == engine equity:", np.allclose(eq_chk.to_numpy(), sim(full(T_C6), weekly, WF_START)["equity"].to_numpy()))

selector == engine (C6): True | (buffer 15): True
ledger equity == engine equity: True


### 1. Trade-level diagnosis of C6 (stitched walk-forward 2022-04 → now, and the last 12 months)

In [3]:
tr, eq_c6, n_open_c6 = ledger(full(T_C6), weekly, WF_START)
log_df = pd.DataFrame(c6_log)
prev_session = pd.Series(idx[:-1], index=idx[1:])
tr["Decision_Exit"] = tr["Exit"].map(prev_session)
tr = tr.merge(log_df.rename(columns={"Date": "Decision_Exit", "Rank": "Exit_Rank", "Score": "Exit_Score"}),
              on=["Decision_Exit", "Symbol"], how="left")
tr["Weeks"] = (tr["Sessions"] / 5).round().clip(lower=1).astype(int)
tr["Hold bucket"] = pd.cut(tr["Weeks"], [0, 1, 3, 8, 10_000], labels=["1 wk", "2–3 wk", "4–8 wk", "9+ wk"])
tr["Below_buy_price_raw"] = tr["Exit_Open"] < tr["Entry_Open"]
tr["Loss_net"] = tr["Return_net_%"] < 0
tr = tr.sort_values("Exit").reset_index(drop=True)
# churn: exit followed by a re-entry of the same stock within 28 calendar days
nxt = tr.sort_values("Entry").groupby("Symbol")[["Entry", "Entry_Open"]]
tr = tr.sort_values(["Symbol", "Entry"])
tr["Next_Entry"] = tr.groupby("Symbol")["Entry"].shift(-1)
tr["Next_Entry_Open"] = tr.groupby("Symbol")["Entry_Open"].shift(-1)
tr["Rebought_4w"] = (tr["Next_Entry"] - tr["Exit"]).dt.days <= 28
# cost of a churn round trip: price given up by selling and re-buying higher (or gained if lower) + 2 × 0.1% cost,
# on the exited position size, in % of portfolio at that time
tr["Churn_cost_%_equity"] = np.where(tr["Rebought_4w"], ((tr["Next_Entry_Open"] / tr["Exit_Open"] - 1) + 2 * be.COST)
                                     * tr["Exit_Value_%_equity"], np.nan)
tr = tr.sort_values("Exit").reset_index(drop=True)
last12 = idx[-1] - pd.DateOffset(years=1)


def diagnose(t, label, n_open=None):
    wins, losses = t[t["Return_net_%"] > 0], t[t["Return_net_%"] <= 0]
    pnl = t["PnL_%_of_equity_at_entry"]
    top = pnl.sort_values(ascending=False).head(max(1, math.ceil(len(t) * 0.1)))
    ch = t[t["Rebought_4w"]]
    out = {"Period": label, "Round trips (closed)": len(t), "Open at end": n_open,
           "% exits below buy price (raw open→open)": t["Below_buy_price_raw"].mean() * 100,
           "% losing trades (net of costs)": t["Loss_net"].mean() * 100,
           "Win rate % (net)": (t["Return_net_%"] > 0).mean() * 100,
           "Avg win %": wins["Return_net_%"].mean(), "Avg loss %": losses["Return_net_%"].mean(),
           "Payoff ratio": wins["Return_net_%"].mean() / abs(losses["Return_net_%"].mean()),
           "Profit factor (portfolio P&L)": pnl[pnl > 0].sum() / abs(pnl[pnl < 0].sum()),
           "Expectancy % per trade": t["Return_net_%"].mean(),
           "Expectancy bp of portfolio per trade": pnl.mean() * 100,
           "Median hold (weeks)": t["Weeks"].median(), "Mean hold (weeks)": t["Weeks"].mean(),
           "Top 10% trades share of net P&L %": top.sum() / pnl.sum() * 100,
           "Top 10% trades share of gross profit %": top.sum() / pnl[pnl > 0].sum() * 100,
           "Exits re-bought within 4 wk": len(ch), "% of exits re-bought within 4 wk": len(ch) / len(t) * 100,
           "Churn: avg re-buy vs sell price %": ((ch["Next_Entry_Open"] / ch["Exit_Open"] - 1) * 100).mean(),
           "Churn cost total (% of portfolio, summed)": ch["Churn_cost_%_equity"].sum(),
           "Churn cost from trading costs only (% of portfolio)": (2 * be.COST * ch["Exit_Value_%_equity"]).sum()}
    return out


periods = [(tr, f"WF {WF_START}→{idx[-1]:%Y-%m-%d}", n_open_c6), (tr[tr["Exit"] >= last12], f"Last 12m (exits since {last12:%Y-%m-%d})", None)]
diag = pd.DataFrame([diagnose(t, lab, no) for t, lab, no in periods]).set_index("Period").T
print(diag.round(2).to_string())


def breakdown(t, by):
    g = t.groupby(by, observed=True)
    return pd.DataFrame({"Trades": g.size(), "% of trades": g.size() / len(t) * 100, "Win rate %": g["Return_net_%"].apply(lambda r: (r > 0).mean() * 100),
                         "Avg return %": g["Return_net_%"].mean(), "Median return %": g["Return_net_%"].median(),
                         "Sum P&L (% of portfolio)": g["PnL_%_of_equity_at_entry"].sum(),
                         "% below buy price (raw)": g["Below_buy_price_raw"].mean() * 100})


tables = {}
for t, lab, _ in periods:
    tables[(lab, "Hold bucket")] = breakdown(t, "Hold bucket")
    tables[(lab, "Exit reason")] = breakdown(t, "Reason")
    for k in ("Hold bucket", "Exit reason"):
        print(f"\n{lab} — by {k}\n", tables[(lab, k)].round(2).to_string())
wk_dist = tr["Weeks"].clip(upper=20).value_counts().sort_index()
print("\nHolding-period distribution (weeks, 20 = 20+):", wk_dist.to_dict())

# naive per-stock view: how often was the sell price below the buy price?
by_stock = tr.groupby("Symbol").agg(Trades=("Return_net_%", "size"), Below_buy_pct=("Below_buy_price_raw", lambda x: x.mean() * 100),
                                     Win_rate_net=("Return_net_%", lambda r: (r > 0).mean() * 100), Avg_return=("Return_net_%", "mean"),
                                     Best=("Return_net_%", "max"), Worst=("Return_net_%", "min"),
                                     Sum_PnL_pct_portfolio=("PnL_%_of_equity_at_entry", "sum"), Median_weeks=("Weeks", "median")).round(2)
by_stock = by_stock.sort_values("Trades", ascending=False)
print(f"\nStocks with >50% of sells below the buy price: {(by_stock.Below_buy_pct > 50).sum()} of {len(by_stock)} "
      f"(median across stocks {by_stock.Below_buy_pct.median():.0f}%); stocks with positive total P&L: {(by_stock.Sum_PnL_pct_portfolio > 0).sum()}")
print(by_stock.head(15).to_string())
mrk = tr[tr.Symbol == "MRK"][["Entry", "Exit", "Entry_Open", "Exit_Open", "Return_net_%", "Weeks", "Reason", "Exit_Rank", "Exit_Score", "Rebought_4w"]]
print("\nMRK round trips (C6, walk-forward run):\n", mrk.round(2).to_string(index=False))

out_cols = ["Symbol", "Entry", "Exit", "Entry_Open", "Exit_Open", "Return_net_%", "PnL_%_of_equity_at_entry", "Sessions", "Weeks",
            "Hold bucket", "Reason", "Exit_Rank", "Exit_Score", "Below_buy_price_raw", "Rebought_4w", "Next_Entry", "Next_Entry_Open",
            "Churn_cost_%_equity"]
tr[out_cols].round(4).to_csv(be.REPORTS_DIR / "trade_diagnostics_v4.csv", index=False)
summ = [diag.reset_index().rename(columns={"index": "Metric"}).assign(Table="summary")]
for (lab, k), tb in tables.items():
    summ.append(tb.reset_index().rename(columns={tb.index.name or "index": "Group"}).assign(Table=f"{lab} | by {k}"))
pd.concat(summ, ignore_index=True).round(3).to_csv(be.REPORTS_DIR / "trade_diagnostics_summary_v4.csv", index=False)
by_stock.to_csv(be.REPORTS_DIR / "trade_diagnostics_by_stock_v4.csv")

Period                                               WF 2022-04-01→2026-09-24  Last 12m (exits since 2025-09-24)
Round trips (closed)                                                   838.00                             194.00
Open at end                                                             10.00                                NaN
% exits below buy price (raw open→open)                                 51.07                              54.64
% losing trades (net of costs)                                          53.94                              57.22
Win rate % (net)                                                        46.06                              42.78
Avg win %                                                               11.16                              14.49
Avg loss %                                                              -7.71                              -7.97
Payoff ratio                                                             1.45                   

### 2–3. Candidates: sector cap and exit rules (walk-forward)

In [4]:
def cand(**kw):
    reb = kw.pop("reb", weekly)
    decide = kw.pop("decide", reb)
    return full(select(decide=decide, **kw)), reb


CANDS = {
    "C6 current (cap 4, weekly top-10)": (full(T_C6), weekly),
    "CAP2 max 2 per sector": cand(max_sector=2), "CAP3 max 3 per sector": cand(max_sector=3),
    "CAP5 max 5 per sector": cand(max_sector=5), "CAP-none (top 10, no sector cap)": cand(max_sector=10),
    "E1 buffer: hold until rank > 15": cand(keep_rank=15), "E2 buffer: hold until rank > 20": cand(keep_rank=20),
    "E3 buffer: hold until rank > 25": cand(keep_rank=25),
    "E4 score exit: hold until score <= 0": cand(keep_rank=np.inf, entry_min=0.0),
    "E5 score exit: hold until score <= 10": cand(keep_rank=np.inf, entry_min=10.0),
    "E6 minimum hold 4 weeks": cand(min_hold=4),
    "E7 every 2 weeks + buffer 20": cand(reb=biweekly, keep_rank=20),
    "E8 monthly + buffer 20": cand(reb=monthly, keep_rank=20),
    "E9 rank entry, exit close < MA50": cand(keep_rank=np.inf, stop="ma50"),
    "E10 rank entry, exit 3xATR trailing stop": cand(keep_rank=np.inf, stop="atr"),
}
THRESH = {x: cand(decide=daily, reb=weekly, keep_rank=np.inf, entry_min=float(x)) for x in (10, 20, 30, 40)}
for x, v in THRESH.items():
    CANDS[f"E11 fixed threshold X={x} (info)"] = v


def quarter_starts(start, end):
    return list(pd.date_range(start, end, freq="QS"))


def walk_forward(pool, train_months=12):
    """Each quarter: the pool member with the best trailing-12m Sharpe is traded for the next quarter (v3 method)."""
    starts = quarter_starts(WF_START, idx[-1])
    target = pd.DataFrame(0.0, index=idx, columns=close_all.columns)
    reb = pd.Series(False, index=idx)
    picks = []
    for i, qs in enumerate(starts):
        tr_s, tr_e = qs - pd.DateOffset(months=train_months), qs - pd.Timedelta(days=1)
        sc = {k: be.metrics(sim(*pool[k], tr_s, tr_e))["Sharpe"] for k in pool}
        best = max(sc, key=lambda k: -np.inf if pd.isna(sc[k]) else sc[k])
        lo = idx.searchsorted(qs) - 1
        hi = idx.searchsorted(starts[i + 1]) - 1 if i + 1 < len(starts) else len(idx)
        t, r = pool[best]
        target.iloc[lo:hi] = t.iloc[lo:hi].values
        reb.iloc[lo:hi] = r.reindex(idx).astype("boolean").fillna(False).astype(bool).iloc[lo:hi].values
        reb.iloc[lo] = True
        picks.append({"quarter": qs.date(), "pick": best})
    return (target, reb), pd.DataFrame(picks)


WF_POOLS = {
    "E-BUF-WF buffer {15,20,25} walk-forward": {k: CANDS[k] for k in CANDS if k.startswith(("E1 ", "E2 ", "E3 "))},
    "E-SC-WF score exit {0,10} walk-forward": {k: CANDS[k] for k in CANDS if k.startswith(("E4 ", "E5 "))},
    "E-STOP-WF MA50 vs 3xATR walk-forward": {k: CANDS[k] for k in CANDS if k.startswith(("E9 ", "E10 "))},
    "E11 absolute threshold, X walk-forward": {f"X={x}": v for x, v in THRESH.items()},
    "CAP-WF cap {2,3,4,5,none} walk-forward (diagnostic)": {k: CANDS[k] for k in CANDS if k.startswith(("C6", "CAP"))},
}
wf_picks = {}
for name, pool in WF_POOLS.items():
    CANDS[name], wf_picks[name] = walk_forward(pool)
    print(name, "picks:", wf_picks[name]["pick"].value_counts().to_dict())

bench = {}
for b in ("QQQ", "SPY"):
    t = pd.DataFrame(0.0, index=idx, columns=close_all.columns); t[b] = 1.0
    bench[f"{b} buy & hold"] = (t, None)

rows, curves = [], {}
for name, (t, r) in {**bench, **CANDS}.items():
    for seg, (s, e) in SEGMENTS.items():
        res = sim(t, r, s, e)
        m = be.metrics(res, name)
        c = res["trades"]
        m.update({"Segment": seg, "Avg win % (closed, net)": c.loc[c.Return > 0, "Return"].mean() * 100 if len(c) else np.nan,
                  "Avg loss % (closed, net)": c.loc[c.Return <= 0, "Return"].mean() * 100 if len(c) else np.nan,
                  "Median hold (sessions)": (c["Exit"] - c["Entry"]).dt.days.median() * 252 / 365 if len(c) else np.nan})
        rows.append(m)
        if seg == "WF stitched":
            curves[name] = res["equity"]
res_df = pd.DataFrame(rows)
res_df.round(4).to_csv(be.REPORTS_DIR / "strategy_comparison_v4.csv", index=False)
SHOW = ["CAGR %", "Total Return %", "Sharpe", "Max DD %", "Turnover x/yr", "Trades", "Win Rate % (closed, net)",
        "Avg win % (closed, net)", "Avg loss % (closed, net)"]
wf = res_df[res_df.Segment == "WF stitched"].set_index("Strategy")
nv = res_df[res_df.Segment == "Never-seen 2022-04→2024-09"].set_index("Strategy")
l12 = res_df[res_df.Segment == "Last 12m"].set_index("Strategy")
table = wf[SHOW].join(nv[["Sharpe", "Max DD %"]].add_prefix("Never-seen ")).join(l12[["Sharpe", "CAGR %"]].add_prefix("Last-12m "))
print(table.round(2).to_string())

E-BUF-WF buffer {15,20,25} walk-forward picks: {'E2 buffer: hold until rank > 20': 8, 'E3 buffer: hold until rank > 25': 6, 'E1 buffer: hold until rank > 15': 4}


E-SC-WF score exit {0,10} walk-forward picks: {'E4 score exit: hold until score <= 0': 14, 'E5 score exit: hold until score <= 10': 4}


E-STOP-WF MA50 vs 3xATR walk-forward picks: {'E10 rank entry, exit 3xATR trailing stop': 13, 'E9 rank entry, exit close < MA50': 5}


E11 absolute threshold, X walk-forward picks: {'X=20': 8, 'X=40': 4, 'X=30': 3, 'X=10': 3}


CAP-WF cap {2,3,4,5,none} walk-forward (diagnostic) picks: {'CAP2 max 2 per sector': 10, 'CAP-none (top 10, no sector cap)': 6, 'CAP3 max 3 per sector': 1, 'C6 current (cap 4, weekly top-10)': 1}


                                                     CAGR %  Total Return %  Sharpe  Max DD %  Turnover x/yr  Trades  Win Rate % (closed, net)  Avg win % (closed, net)  Avg loss % (closed, net)  Never-seen Sharpe  Never-seen Max DD %  Last-12m Sharpe  Last-12m CAGR %
Strategy                                                                                                                                                                                                                                                                   
QQQ buy & hold                                        18.11          110.10    0.85    -29.07           0.22       1                       NaN                      NaN                       NaN               0.61               -29.07             1.18            24.06
SPY buy & hold                                        14.03           79.58    0.85    -21.29           0.22       1                       NaN                      NaN                       NaN   

### 4. Decision (pre-declared rule)

In [5]:
BASE = "C6 current (cap 4, weekly top-10)"
b = table.loc[BASE]
qual = []
for name in CANDS:
    if name == BASE or "(info)" in name or "(diagnostic)" in name:
        continue
    r = table.loc[name]
    beats = r["Sharpe"] > b["Sharpe"] and r["Max DD %"] >= b["Max DD %"] - 2.0 and r["Never-seen Sharpe"] >= b["Never-seen Sharpe"]
    close = (not beats and r["Turnover x/yr"] <= 0.75 * b["Turnover x/yr"] and r["Sharpe"] >= b["Sharpe"] - 0.03
             and r["Max DD %"] > b["Max DD %"])
    qual.append({"Candidate": name, "WF Sharpe": r["Sharpe"], "WF Max DD %": r["Max DD %"], "Never-seen Sharpe": r["Never-seen Sharpe"],
                 "Turnover x/yr": r["Turnover x/yr"], "Qualifies (switch)": beats, "Close call": close})
qual = pd.DataFrame(qual).sort_values("WF Sharpe", ascending=False)
winners = qual[qual["Qualifies (switch)"]]
DECISION = winners.iloc[0]["Candidate"] if len(winners) else BASE
print(f"C6: Sharpe {b['Sharpe']:.3f}, MaxDD {b['Max DD %']:.1f}%, never-seen Sharpe {b['Never-seen Sharpe']:.3f}, turnover {b['Turnover x/yr']:.1f}x")
print("DECISION:", "switch to " + DECISION if DECISION != BASE else "keep C6 (no candidate met the rule)")
print("Close calls:", qual.loc[qual["Close call"], "Candidate"].tolist())
print(qual.round(3).to_string(index=False))

C6: Sharpe 1.471, MaxDD -28.6%, never-seen Sharpe 0.924, turnover 39.8x
DECISION: switch to CAP2 max 2 per sector
Close calls: []
                               Candidate  WF Sharpe  WF Max DD %  Never-seen Sharpe  Turnover x/yr  Qualifies (switch)  Close call
        CAP-none (top 10, no sector cap)       1.64       -30.28               0.82          39.59               False       False
                   CAP2 max 2 per sector       1.48       -23.66               1.06          36.48                True       False
                   CAP5 max 5 per sector       1.43       -30.36               0.78          40.84               False       False
         E2 buffer: hold until rank > 20       1.33       -29.95               0.82          24.03               False       False
                   CAP3 max 3 per sector       1.33       -29.76               0.66          39.59               False       False
            E7 every 2 weeks + buffer 20       1.28       -27.91               0.95 

### 5. Charts

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
colors = {"rank drift (score > 0)": "#2563eb", "sector cap displacement": "#d97706", "score <= 0": "#b91c1c", "not eligible / no data": "#6b7280"}
ax = axes[0]
for reason, g in tr.groupby("Reason"):
    ax.scatter(g["Weeks"] + np.random.default_rng(0).uniform(-0.25, 0.25, len(g)), g["Return_net_%"], s=10, alpha=0.55,
               color=colors.get(reason, "#6b7280"), label=f"{reason} ({len(g)})")
ax.axhline(0, color="k", lw=0.8); ax.set_xscale("symlog", linthresh=10); ax.set_xlabel("Holding period (weeks)"); ax.set_ylabel("Trade return % (net)")
ax.set_title(f"C6 (cap 4) round trips {WF_START[:7]}→now: holding period vs return", fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1]
hb = tables[(periods[0][1], "Hold bucket")]
ax.bar(hb.index.astype(str), hb["Sum P&L (% of portfolio)"], color=["#b91c1c" if v < 0 else "#15803d" for v in hb["Sum P&L (% of portfolio)"]])
for i, (n_, w_) in enumerate(zip(hb["Trades"], hb["Win rate %"])):
    ax.text(i, 0, f"{n_} trades\nwin {w_:.0f}%", ha="center", va="bottom", fontsize=8)
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("Summed P&L (% of portfolio at entry)"); ax.set_title("P&L by holding period", fontsize=10); ax.grid(alpha=0.3, axis="y")
ax = axes[2]
er = tables[(periods[0][1], "Exit reason")]
ax.barh(er.index, er["Sum P&L (% of portfolio)"], color=[colors.get(i, "#6b7280") for i in er.index])
ax.axvline(0, color="k", lw=0.8); ax.set_title("P&L by exit reason", fontsize=10); ax.grid(alpha=0.3, axis="x")
for i, (n_, w_) in enumerate(zip(er["Trades"], er["Win rate %"])):
    ax.text(0, i, f" {n_} trades, win {w_:.0f}%", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(be.REPORTS_DIR / "trade_diagnostics_v4.png", dpi=120)

show = [BASE, "CAP3 max 3 per sector", "CAP-none (top 10, no sector cap)", "E2 buffer: hold until rank > 20", "E4 score exit: hold until score <= 0",
        "E6 minimum hold 4 weeks", "E8 monthly + buffer 20", "E10 rank entry, exit 3xATR trailing stop", "E11 absolute threshold, X walk-forward",
        "QQQ buy & hold", "SPY buy & hold"]
show = list(dict.fromkeys(show + ([DECISION] if DECISION != BASE else [])))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
for n_ in show:
    e_ = curves[n_]
    st = (dict(lw=2.6, color="k") if n_ == BASE else dict(lw=2.4, color="#dc2626") if n_ == DECISION
          else dict(lw=1.1, ls="--" if "buy & hold" in n_ else "-", alpha=0.85))
    if n_ == DECISION and n_ != BASE:
        n_label = f"{n_} = NEW LIVE RULE"
    else:
        n_label = n_
    a1.plot(e_.index, e_.values, label=f"{n_label} ({(e_.iloc[-1] - 1) * 100:+.0f}%, Sharpe {table.at[n_, 'Sharpe']:.2f}, DD {table.at[n_, 'Max DD %']:.0f}%)", **st)
    a2.plot(e_.index, (e_ / e_.cummax().clip(lower=1) - 1) * 100, lw=2 if n_ in (BASE, DECISION) else 0.9,
            **({"color": "k"} if n_ == BASE else {"color": "#dc2626"} if n_ == DECISION else {"alpha": 0.6}))
for a in (a1, a2):
    a.axvspan(pd.Timestamp(WF_START), pd.Timestamp(SEEN_START), color="#e0f2fe", alpha=0.5); a.grid(alpha=0.3)
a1.set_yscale("log"); a1.set_ylabel("Equity (log, start = 1.0)"); a1.legend(fontsize=7.5, loc="upper left")
a1.set_title("v4 walk-forward 2022-04 → now (blue band = never-seen). Hand-picked universe → selection bias", fontsize=10)
a2.set_ylabel("Drawdown %")
fig.tight_layout()
fig.savefig(be.REPORTS_DIR / "strategy_walkforward_v4.png", dpi=120)
print("Saved strategy_comparison_v4.csv, trade_diagnostics_v4.csv (+ summary, by_stock), trade_diagnostics_v4.png, strategy_walkforward_v4.png")

Saved strategy_comparison_v4.csv, trade_diagnostics_v4.csv (+ summary, by_stock), trade_diagnostics_v4.png, strategy_walkforward_v4.png
